In [1]:
from dotenv import load_dotenv
load_dotenv()

import os
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import SupabaseVectorStore
from supabase.client import Client, create_client

supabase_url = os.getenv("SUPABASE_URL")
supabase_key = os.getenv("SUPABASE_PRIVATE_KEY")
supabase: Client = create_client(supabase_url, supabase_key)

embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004",google_api_key=os.getenv("GOOGLE_API_KEY"))

I0000 00:00:1731341096.992036 88397133 check_gcp_environment_no_op.cc:29] ALTS: Platforms other than Linux and Windows are not supported


In [2]:
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain.schema import Document

loader = UnstructuredPDFLoader("real-analysis/real-analysis.pdf", mode="elements")
documents = loader.load()

In [12]:
textbook_id = "e8944344-2248-472d-9a8c-56a85c76bcba"
start_page = 17 # inclusive
end_page = 418 # inclusive

docs_dict = {} # creating dictionary, for each page
for doc in documents:
    page_number = doc.metadata["page_number"]
    if (docs_dict.get(page_number)):
        docs_dict[page_number].append(doc.page_content)
    else:
        docs_dict[page_number] = [doc.page_content]

# adding all elements on pages into one document
new_docs = []
for page_number, docs in docs_dict.items():
    combined_docs = str(" ".join(docs))
    if (page_number >= start_page) and (page_number <= end_page):
        new_docs.append(Document(page_content=combined_docs, metadata = {"id": textbook_id, "interval": page_number - start_page + 1, "type": "textbook"}))

print(new_docs)

[Document(metadata={'id': 'e8944344-2248-472d-9a8c-56a85c76bcba', 'interval': 1, 'type': 'textbook'}, page_content='C01 12/08/2010 12:3:14 Page 1 CHAPTER 1 PRELIMINARIES In this initial chapter we will present the background needed for the study of real analysis. Section 1.1 consists of a brief survey of set operations and functions, two vital tools for all of mathematics. In it we establish the notation and state the basic deﬁnitions and properties that will be used throughout the book. We will regard the word ‘‘set’’ as synonymous with the words ‘‘class,’’ ‘‘collection,’’ and ‘‘family,’’ and we will not deﬁne these terms or give a list of axioms for set theory. This approach, often referred to as ‘‘naive’’ set theory, is quite adequate for working with sets in the context of real analysis. Section 1.2 is concerned with a special method of proof called Mathematical Induction. It is related to the fundamental properties of the natural number system and, though it is restricted to provi

In [13]:
# adding to vector store. Will automatically update
vector_store = SupabaseVectorStore.from_documents(
    new_docs,
    embeddings,
    client=supabase,
    table_name="documents",
    query_name="match_documents"
)